# 04 · Comparison with MiniSat

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

In [ ]:
!pip install python-sat -q

In [ ]:
import pickle
import random
import time
import torch
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from pysat.solvers import Minisat22

from neurosat import NeuroSAT, make_batch, to_device

## Settings

In [ ]:
SEED = 42
DIM = 128
ROUND_LIST = [26, 128]

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load data and model

In [ ]:
with open("test.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

datasets = {"10–40": test_data}
for n, data in large_test_data.items():
    datasets[str(n)] = data

model = NeuroSAT(DIM).to(device)
model.load_state_dict(torch.load("neurosat.pt", map_location=device))
model.eval()

for name, data in datasets.items():
    print(f"n={name}: {sum(p['label'] for p in data)} SAT formulas")

## MiniSat

In [ ]:
def run_minisat(problem):
    start = time.perf_counter()
    with Minisat22(bootstrap_with=problem["clauses"]) as solver:
        result = solver.solve()
        stats = solver.accum_stats()
    elapsed = time.perf_counter() - start
    return result, elapsed, stats["decisions"], stats["conflicts"]

## NeuroSAT

In [ ]:
def satisfies(clauses, assignment):
    for clause in clauses:
        if not any(assignment[abs(lit) - 1] == (lit > 0) for lit in clause):
            return False
    return True


def run_neurosat(problem, rounds):
    start = time.perf_counter()

    batch = to_device(make_batch([problem]), device)
    with torch.no_grad():
        logit, L_h = model(batch, rounds, return_literals=True)

    solved = False
    if logit.item() > 0:
        clusters = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(L_h.cpu().numpy())
        n = problem["n_vars"]
        for true_cluster in [0, 1]:
            assignment = [clusters[i] == true_cluster for i in range(n)]
            if satisfies(problem["clauses"], assignment):
                solved = True
                break

    elapsed = time.perf_counter() - start
    return solved, elapsed

## Warm up

In [ ]:
for problem in test_data[:10]:
    run_minisat(problem)
    run_neurosat(problem, ROUND_LIST[0])

## Run on SAT formulas

In [ ]:
results = {}

for name, data in datasets.items():
    sat_problems = [p for p in data if p["label"] == 1]

    minisat_times = []
    minisat_decisions = []
    minisat_conflicts = []
    for problem in sat_problems:
        _, elapsed, decisions, conflicts = run_minisat(problem)
        minisat_times.append(elapsed)
        minisat_decisions.append(decisions)
        minisat_conflicts.append(conflicts)

    results[name] = {
        "minisat_time": sum(minisat_times) / len(minisat_times),
        "minisat_decisions": sum(minisat_decisions) / len(minisat_decisions),
        "minisat_conflicts": sum(minisat_conflicts) / len(minisat_conflicts),
        "minisat_solved": 1.0,
    }

    for rounds in ROUND_LIST:
        solved = []
        times = []
        for problem in sat_problems:
            s, elapsed = run_neurosat(problem, rounds)
            solved.append(s)
            times.append(elapsed)
        results[name][f"neurosat_{rounds}_time"] = sum(times) / len(times)
        results[name][f"neurosat_{rounds}_solved"] = sum(solved) / len(solved)

    print(f"Done n={name}")

## Results table

In [ ]:
header = f"{'n':>6s} | {'MiniSat ms':>10s} | {'decisions':>9s} | {'conflicts':>9s}"
for rounds in ROUND_LIST:
    header += f" | {'NS' + str(rounds) + ' ms':>9s} | {'NS' + str(rounds) + ' solved':>10s}"
print(header)

for name, r in results.items():
    row = f"{name:>6s} | {r['minisat_time'] * 1000:10.2f} | {r['minisat_decisions']:9.1f} | {r['minisat_conflicts']:9.1f}"
    for rounds in ROUND_LIST:
        row += f" | {r[f'neurosat_{rounds}_time'] * 1000:9.2f} | {r[f'neurosat_{rounds}_solved']:10.3f}"
    print(row)

## Plots

In [ ]:
names = list(results.keys())
x = range(len(names))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([i - width for i in x], [results[n]["minisat_time"] * 1000 for n in names], width, label="MiniSat")
for k, rounds in enumerate(ROUND_LIST):
    axes[0].bar([i + k * width for i in x], [results[n][f"neurosat_{rounds}_time"] * 1000 for n in names], width, label=f"NeuroSAT {rounds} rounds")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(names)
axes[0].set_yscale("log")
axes[0].set_xlabel("Variables")
axes[0].set_ylabel("Mean time per formula (ms)")
axes[0].legend()

axes[1].bar([i - width for i in x], [results[n]["minisat_solved"] for n in names], width, label="MiniSat")
for k, rounds in enumerate(ROUND_LIST):
    axes[1].bar([i + k * width for i in x], [results[n][f"neurosat_{rounds}_solved"] for n in names], width, label=f"NeuroSAT {rounds} rounds")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(names)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("Variables")
axes[1].set_ylabel("Share of SAT formulas solved")
axes[1].legend()

plt.tight_layout()
plt.savefig("minisat_comparison.png", dpi=150)
plt.show()

## Save

In [ ]:
with open("minisat_comparison.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved minisat_comparison.pkl and minisat_comparison.png")